In [ ]:
# Path to directory with your images
image_directory = "/content/gdrive/MyDrive/face_dataset/input"
# Replace with your name
person_name = "person_01"
# Create a directory in your google drive to store the face images extracted
OUTPUT_DIR = "/content/gdrive/MyDrive/face_dataset/cropped/"


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')


In [ ]:
import cv2
import os

In [ ]:
def save_cropped_face(image, output_dir, person_name, image_count):
    """Save the cropped face as an image in the appropriate directory."""
    #person_dir = os.path.join(output_dir, person_name)
    person_dir = output_dir
    os.makedirs(person_dir, exist_ok=True)
    face_filename = os.path.join(person_dir, f"{person_name}_{image_count}.jpg")
    cv2.imwrite(face_filename, image)
    print(f"Saved: {face_filename}")

In [ ]:
def detect_faces_in_image(image_path, person_name, output_dir, global_image_count):
    """Detect faces in an image and save cropped faces to the database."""
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

    # Read the image
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error: Unable to read image {image_path}")
        return global_image_count

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    print(faces)
    if len(faces) == 0:
        print(f"No faces detected in {image_path}")
        return global_image_count

    for (x, y, w, h) in faces:
        face = image[y:y+h, x:x+w]  # Crop the face
        save_cropped_face(face, output_dir, person_name, global_image_count)
        global_image_count += 1

    print(f"Processed {len(faces)} faces from {image_path}")
    return global_image_count


In [ ]:
def process_image_directory(image_dir, person_name, output_dir=OUTPUT_DIR):
    """Process all images in a directory and detect faces."""
    if not os.path.exists(image_dir):
        print(f"Error: Directory {image_dir} does not exist.")
        return

    image_files = [
        os.path.join(image_dir, f)
        for f in os.listdir(image_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))
    ]

    if not image_files:
        print(f"No image files found in {image_dir}")
        return

    print(f"Processing directory: {image_dir}...")
    global_image_count = 0  # Use a global counter for all images
    for image_file in image_files:
        global_image_count = detect_faces_in_image(image_file, person_name, output_dir, global_image_count)


In [ ]:
process_image_directory(image_directory, person_name)